# Jazmine Tabora - DSCI 552 HW 3

### 1. Time Series Classification Part 1: Feature Creation/Extraction

#### (a) Download data

#### (b) Keep datasets 1 and 2 in folders bending1 and bending 2, as well as datasets 1, 2, and 3 in other folders as test data and other datasets as train data.

In [8]:
from pathlib import Path
import pandas as pd

# Names of the sensor columns in each file
columns = [
    "time", "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23"
]

arem_folder = Path("AReM")
instances = []

# Reading each dataset file
for file_path in arem_folder.rglob("dataset*.csv"):
    activity = file_path.parent.name
    dataset_number = int(file_path.stem.replace("dataset", ""))

    # Skipping the metadata lines and handle comma and space separated values
    data = pd.read_csv(
    file_path,
    skiprows=5,
    sep=r"[,\s]+",
    header=None,
    names=columns,
    usecols=range(7),
    engine="python"
)

    # Following the test-set rule 
    if activity in ["bending1", "bending2"]:
        split = "test" if dataset_number in [1, 2] else "train"
    else:
        split = "test" if dataset_number in [1, 2, 3] else "train"

    instances.append({
        "activity": activity,
        "dataset": dataset_number,
        "split": split,
        "data": data
    })

# Checking for the number of instances and the train/test split
print("Total instances:", len(instances))
print(pd.Series([item["split"] for item in instances]).value_counts())
print("First instance shape:", instances[0]["data"].shape)

Total instances: 88
train    69
test     19
Name: count, dtype: int64
First instance shape: (480, 7)


#### (c) Feature Extraction

##### i. Research what types of time-domain features are usually used in time series classification and list them (examples are minimum, maximum, mean, etc).

* Minimum: smallest reading
* Maximum: biggest reading
* Mean: average
* Median: middle reading after the values are sorted
* Standard deviation: how spread out the readings are from the mean
* First and third quartiles: point where about 25% of the readings are lower and point where about 75% of the readings are lower
* Range: maximum minus the minimum
* Interquartile range: third quartile minus the first quartile.
* Root mean square (RMS): a summary of the typical size of the readings
* Skewness: shows whether the readings lean more toward higher or lower values
* Kurtosis: how much the readings tend to include unusually extreme values
* Zero-crossing rate: how often the readings cross zero


##### ii. Extract the time-domain features minimum, maximum, mean, median, standard deviation, first quartile, and third quartile for all of the 6 time series in each instance. You are free to normalize/standardize features or use them directly.

In [10]:
import pandas as pd

# Six sensor measurements
sensor_columns = [
    "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23"
]

feature_rows = []

# One feature table row for each instance
for instance_number, item in enumerate(instances, start=1):
    row = {
        "instance": instance_number,
        "activity": item["activity"],
        "dataset": item["dataset"],
        "split": item["split"]
    }

    # Calculating the seven features for each sensor
    for sensor in sensor_columns:
        readings = item["data"][sensor]

        row[f"min_{sensor}"] = readings.min()
        row[f"max_{sensor}"] = readings.max()
        row[f"mean_{sensor}"] = readings.mean()
        row[f"median_{sensor}"] = readings.median()
        row[f"std_{sensor}"] = readings.std()
        row[f"q1_{sensor}"] = readings.quantile(0.25)
        row[f"q3_{sensor}"] = readings.quantile(0.75)

    feature_rows.append(row)

# Putting all 88 instances in a table
feature_table = pd.DataFrame(feature_rows)

print("Feature table size:", feature_table.shape)
feature_table.head()

Feature table size: (88, 46)


,instance,activity,dataset,split,min_avg_rss12,max_avg_rss12,mean_avg_rss12,median_avg_rss12,std_avg_rss12,q1_avg_rss12,...,std_avg_rss23,q1_avg_rss23,q3_avg_rss23,min_var_rss23,max_var_rss23,mean_var_rss23,median_var_rss23,std_var_rss23,q1_var_rss23,q3_var_rss23
0,1,bending1,1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,2,bending1,2,test,38.00,45.67,42.812812,42.50,1.435550,42.00,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,3,bending1,3,train,35.00,47.40,43.954500,44.33,1.558835,43.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,4,bending1,4,train,33.00,47.75,42.179812,43.50,3.670666,39.15,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,5,bending1,5,train,33.00,45.75,41.678063,41.75,2.243490,41.33,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


##### iii. Estimate the standard deviation of each of the time-domain features you extracted from the data. Then, use Python’s bootstrapped or any other method to build a 90% bootsrap confidence interval for the standard deviation of each feature.

In [12]:
import numpy as np
import pandas as pd

# Selecting the 42 calculated features
# leaving out the four identifying columns
feature_columns = [
    column for column in feature_table.columns
    if column not in ["instance", "activity", "dataset", "split"]
]

rng = np.random.default_rng(42)
bootstrap_results = []

# Calculating the standard deviation and 90% bootstrap interval for each feature
for feature in feature_columns:
    values = feature_table[feature].dropna().to_numpy()

    estimated_sd = np.std(values, ddof=1)
    bootstrap_sds = []

    # Repeatedly resample and calculating each sample's SD
    for _ in range(1000):
        sample = rng.choice(values, size=len(values), replace=True)
        bootstrap_sds.append(np.std(sample, ddof=1))

    # 5th and 95th percentiles for a 90% interval
    lower_bound, upper_bound = np.percentile(bootstrap_sds, [5, 95])

    bootstrap_results.append({
        "feature": feature,
        "estimated_sd": estimated_sd,
        "90%_CI_lower": lower_bound,
        "90%_CI_upper": upper_bound
    })

sd_ci_table = pd.DataFrame(bootstrap_results)
sd_ci_table.head()

,feature,estimated_sd,90%_CI_lower,90%_CI_upper
0,min_avg_rss12,9.569975,8.194325,10.848390
1,max_avg_rss12,4.394362,3.328131,5.267411
2,mean_avg_rss12,5.335718,4.666422,5.871471
3,median_avg_rss12,5.440054,4.800392,5.977818
4,std_avg_rss12,1.772153,1.562906,1.948855


##### iv. Use your judgement to select the three most important time-domain features (one option may be min, mean, and max).

##### I think the three most important time-domain features are the minimum, mean, and maximum. The mean shows a sensor's average/typical reading. The minimum and maximum show the lowest and highest reading respectively. However, it's hard to tell if they are the best three without testing a classifier.

#### 2. ISLR 3.7.4
QUESTION
* 100 observations
* one predictor X
* a numberical response Y
* fit a linear and cubic model which can curve


##### ANSWER
* (a) If the true relationship between X and Y is linear then the cubic modeL's training RSS should be less than or equal to the linear model’s because it can fit a straight line too.

* (b) The linear model is expected to do better because cubic models can bend and curve and may follow noise in the training data. This may make the cubic model look better on the training data but in the test data the noise is random and won't appear in the same places. Therefore, the linear model is expected to have  a lower test RSS than the cube model.

* (c) If the true relationship is nonlinear, then the cubic model’s training RSS should again be less than or equal to the linear model’s because it is more flexible. A cubic model has more options for fitting the training points since it has other terms that make it curve. Cubic model can be linear if their extra terms are set to zero but since it has that flexibility it is more likely to fir better than a linear model if the true relationship isn't linear.

* (d) If the true relationship is nonlinear, then there isn’t enough information to know which model will have a lower test RSS. It depends on how well the cubic model matches the true relationship and whether it overfits.